# 05 — Temporalidade do uso mensal: ausente em 2025, zero em 2026

**Dimensão:** temporalidade (ver `AGENTS.md`, seção "Qualidade de dados").

**Objetivo:** ver se o mês sem treino é registrado do mesmo jeito em 2025 e em 2026, antes de comparar médias.

**Pedido:** em `uso_mensal.csv`, para cada mês de jan/2025 a set/2026:
1. contar as linhas e as linhas com `minutos = 0`;
2. calcular a média e a mediana de minutos;
3. explicar o que muda na virada de 2025 para 2026 e o efeito disso numa média calculada sobre o arquivo como está.

**Referência (roteiro do professor):** 2025 sem nenhum mês com linha zerada; dez/2025 com 13.624 linhas; jan/2026 com 41.130 linhas, 25.697 zeradas; a média cai de 265 para 118 minutos na virada só por isso; contando só quem treinou, jan/2026 dá 314.

Nada em `dados/brutos/` é alterado.

In [ ]:
import pandas as pd
from pathlib import Path

BRUTOS = Path('../dados/brutos')
uso = pd.read_csv(BRUTOS / 'uso_mensal.csv', dtype={'user_id': str, 'mes': str})
print(f'{len(uso):,} linhas | meses: {uso["mes"].min()} a {uso["mes"].max()} ({uso["mes"].nunique()})')
assert not uso.duplicated(['user_id', 'mes']).any(), 'esperado uma linha por usuário e mês'

## 1 e 2. Linhas, zeros, média e mediana por mês

In [ ]:
por_mes = uso.groupby('mes').agg(
    linhas=('minutos', 'size'),
    linhas_zeradas=('minutos', lambda s: (s == 0).sum()),
    media=('minutos', 'mean'),
    mediana=('minutos', 'median'),
)
por_mes['media_so_quem_treinou'] = uso[uso['minutos'] > 0].groupby('mes')['minutos'].mean()
por_mes = por_mes.round(1)
por_mes

## Conferência com o roteiro

In [ ]:
ano = por_mes.index.str[:4]
assert (por_mes.loc[ano == '2025', 'linhas_zeradas'] == 0).all(), '2025: nenhum mês com linha zerada'
assert por_mes.loc['2025-12', 'linhas'] == 13_624
assert por_mes.loc['2026-01', 'linhas'] == 41_130 and por_mes.loc['2026-01', 'linhas_zeradas'] == 25_697
assert round(por_mes.loc['2025-12', 'media']) == 265 and round(por_mes.loc['2026-01', 'media']) == 118
assert round(por_mes.loc['2026-01', 'media_so_quem_treinou']) == 314
print('Números do roteiro conferidos.')

## 3. O que muda na virada

O arquivo muda de regra de registro em janeiro de 2026, sem aviso no dicionário:

- **2025:** só tem linha quem treinou no mês. Mês sem treino = **linha ausente**.
- **2026:** o mês sem treino passa a ter **linha com zero**.

Uma média sobre o arquivo como está divide por mais gente em 2026: os zeros entram no denominador. A média despenca de 265 para 118 minutos na virada sem que ninguém tenha treinado menos. Contando só quem treinou, jan/2026 dá 314, acima de dez/2025.

In [ ]:
comparacao = por_mes[['media', 'media_so_quem_treinou']].rename(columns={
    'media': 'média do arquivo como está',
    'media_so_quem_treinou': 'média só de quem treinou (comparável)',
})
comparacao.loc['2025-10':'2026-03']

In [ ]:
# Quem treinou: a série contínua por trás do salto de linhas
treinou = por_mes['linhas'] - por_mes['linhas_zeradas']
treinou.loc['2025-10':'2026-03'].rename('linhas com treino')

## Além do roteiro: o que tem nas linhas zeradas

Uma linha zerada não diz só "não treinou". As outras colunas mostram se a pessoa ao menos abriu o app.

In [ ]:
zeradas = uso[uso['minutos'] == 0]
abriu_app = (zeradas['sessoes_app'] > 0)
print(f'Linhas zeradas: {len(zeradas):,}')
print(f'  abriram o app sem treinar (sessoes_app > 0): {abriu_app.sum():,} ({abriu_app.mean():.0%})')
print(f'  nem abriram o app:                           {(~abriu_app).sum():,}')
assert (zeradas[['atividades', 'dias_ativos']] == 0).all().all(), 'linha zerada sem atividade e sem dia ativo'

Em 2026, 86% das linhas zeradas são de quem abriu o app e não treinou. Esse comportamento ("entrou, olhou e saiu") é um candidato natural a sinal de churn, e **não existe em 2025**: lá, a mesma pessoa simplesmente não tem linha.

## Conclusão

- **Roteiro conferido:** nenhuma linha zerada em 2025; dez/2025 com 13.624 linhas; jan/2026 com 41.130 linhas, 25.697 zeradas; média de 265 para 118 na virada; 314 contando só quem treinou.
- **Causa:** mudança de regra de registro na virada do ano. Em 2025, mês sem treino é linha ausente; em 2026, é linha zerada.
- **Regra para o grupo:** comparar médias entre os anos só com quem treinou (`minutos > 0`). Ao montar variáveis por usuário e mês, **linha ausente em 2025 vale zero**.
- **Consequência para o modelo (objetivo 4):** com separação no tempo, treinar em 2025 e testar em 2026 com variáveis calculadas sobre o arquivo como está faria o modelo aprender a mudança de registro, não o comportamento. Inatividade precisa ser calculada do mesmo jeito nos dois anos.
- **Sinal novo:** 86% das linhas zeradas de 2026 são de quem abriu o app sem treinar. Esse sinal só existe a partir de 2026 e não pode ser reconstruído para 2025.
- **Próximos passos (fora deste pedido):** definir o universo de cada mês (quem deveria ter linha) com `assinaturas.csv`/`eventos_assinatura.csv` para completar 2025 com zeros; conferir `minutos` contra a soma de `duracao_min` em `atividades.parquet` (2026).

Decisões registradas em `decisoes.md`.